# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/24f2001824/ml-flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
!git clone https://github.com/24f2001824/ml-flyrank.git


df = pd.read_csv("/content/ml-flyrank/data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

fatal: destination path 'ml-flyrank' already exists and is not an empty directory.
Rows: 30000
Columns: 44


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I will rank pages that have higher visibility, are older, have a better position opportunity, and have a smaller word count higher in the queue.

The score will use visibility, freshness risk, position opportunity and depth gap.

Reason codes:
- stale_visible_page
- declining_with_demand
- thin_visible_page
- page_one_decay_risk
- low_ctr_visible_page
- low_engagement_visible_page
- general_refresh_review

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
reason_codes = [
    "stale_visible_page",
    "declining_with_demand",
    "thin_visible_page",
    "page_one_decay_risk",
    "low_ctr_visible_page",
    "low_engagement_visible_page",
    "general_refresh_review"
]

print(reason_codes)

['stale_visible_page', 'declining_with_demand', 'thin_visible_page', 'page_one_decay_risk', 'low_ctr_visible_page', 'low_engagement_visible_page', 'general_refresh_review']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The score combines visibility, freshness, position and content depth. I will use the score to rank all pages and create the baseline action queue.

In [9]:
def percentile_rank(x):
    return x.fillna(0).rank(pct=True)

def normalize(x):
    x = x.fillna(0)
    min_value = x.min()
    max_value = x.max()
    if max_value == min_value:
        return pd.Series(0, index=x.index)
    return (x - min_value) / (max_value - min_value)

df["visibility_score"] = percentile_rank(
    np.log1p(df["impressions_90d"].fillna(0))
)

df["freshness_risk_score"] = percentile_rank(
    df["days_since_last_update"].fillna(0)
)

df["position_opportunity_score"] = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"]
    * (df["avg_position"].fillna(0) > 0).astype(int)
)

df["depth_gap_score"] = (
    1 - percentile_rank(df["word_count"])
) * df["visibility_score"]

df["baseline_action_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).fillna(0).clip(0, 1)

df["rank"] = (
    df["baseline_action_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

output_dir = Path("/content/ml-flyrank/work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

queue = df.sort_values("rank")

queue[
    [
        "content_id",
        "client_id",
        "rank",
        "baseline_action_score",
        "visibility_score",
        "freshness_risk_score",
        "position_opportunity_score",
        "depth_gap_score"
    ]
].to_csv(
    output_dir / "baseline_action_score.csv",
    index=False
)

print("Saved:", output_dir / "baseline_action_score.csv")
print("Top score:", round(queue["baseline_action_score"].max(), 3))

Saved: /content/ml-flyrank/work/outputs/baseline_action_score.csv
Top score: 0.941


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20 pages are reviewed using their score and the observed signals behind the score.

The confidence note reflects how much evidence is available in the data. A recommendation can still be wrong if the observed signals do not reflect the actual content quality or business priority.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def get_reasons(row):
    reasons = []

    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")

    if row["trend_direction"].lower() == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")

    if row["word_count"] > 0 and row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")

    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        reasons.append("page_one_decay_risk")

    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")

    if row["sessions_90d"] >= 30 and (
        (row["engagement_rate"] > 0 and row["engagement_rate"] < 30)
        or (row["scroll_rate"] > 0 and row["scroll_rate"] < 30)
    ):
        reasons.append("low_engagement_visible_page")

    if not reasons:
        reasons.append("general_refresh_review")

    return "|".join(reasons)


def get_action(reasons):
    reasons = set(reasons.split("|"))

    if "thin_visible_page" in reasons:
        return "expand_and_refresh"

    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"

    if "stale_visible_page" in reasons or "declining_with_demand" in reasons:
        return "refresh"

    return "monitor"


top20 = queue.head(20).copy()

top20["reason_codes"] = top20.apply(get_reasons, axis=1)
top20["action"] = top20["reason_codes"].apply(get_action)
top20["confidence_note"] = "Medium: based on observed search and content signals"
top20["what_would_make_it_wrong"] = (
    "The page may have other business or content context not present in the dataset"
)

display(
    top20[
        [
            "content_id",
            "rank",
            "baseline_action_score",
            "action",
            "reason_codes",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

,content_id,rank,baseline_action_score,action,reason_codes,confidence_note,what_would_make_it_wrong
21565,content_9532f197bbc8,1,0.941189,refresh,declining_with_demand|page_one_decay_risk|low_...,Medium: based on observed search and content s...,The page may have other business or content co...
4644,content_4d1fe5b32dc2,2,0.934889,monitor,page_one_decay_risk|low_engagement_visible_page,Medium: based on observed search and content s...,The page may have other business or content co...
18954,content_07f2e7a6f38a,3,0.934080,monitor,page_one_decay_risk|low_engagement_visible_page,Medium: based on observed search and content s...,The page may have other business or content co...
17400,content_e5ae436f9a16,4,0.933606,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,Medium: based on observed search and content s...,The page may have other business or content co...
9348,content_3430a8b94511,5,0.933559,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,Medium: based on observed search and content s...,The page may have other business or content co...
25409,content_cbd93118300b,6,0.933263,refresh_and_review_ctr,declining_with_demand|page_one_decay_risk|low_...,Medium: based on observed search and content s...,The page may have other business or content co...
18458,content_9c195417f6ef,7,0.932991,monitor,page_one_decay_risk|low_engagement_visible_page,Medium: based on observed search and content s...,The page may have other business or content co...
13306,content_ba2acb4ebd04,8,0.931623,monitor,page_one_decay_risk|low_engagement_visible_page,Medium: based on observed search and content s...,The page may have other business or content co...
28354,content_79b25654070a,9,0.931363,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,Medium: based on observed search and content s...,The page may have other business or content co...
8275,content_adddad39251c,10,0.931124,monitor,page_one_decay_risk|low_engagement_visible_page,Medium: based on observed search and content s...,The page may have other business or content co...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some high-ranked pages may not actually need a refresh because the score only uses the available search and content signals.

I will check the lowest-scoring pages in the top 20 and make sure the score does not use the declining label or future performance windows.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
weak_picks = top20.tail(5)

display(
    weak_picks[
        [
            "content_id",
            "rank",
            "baseline_action_score",
            "impressions_90d",
            "avg_position",
            "content_age_days",
            "days_since_last_update",
            "word_count",
            "trend_direction"
        ]
    ]
)

score_fields = [
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "word_count"
]

print("Fields used in score:")
print(score_fields)

print("\nLabel used in score:", "is_declining_label" in score_fields)
print("Trend direction used in score:", "trend_direction" in score_fields)

print("\nFuture-window fields used in score:")
print([
    field for field in [
        "impressions_next_30d",
        "clicks_next_30d",
        "sessions_next_30d"
    ]
    if field in score_fields
])

,content_id,rank,baseline_action_score,impressions_90d,avg_position,content_age_days,days_since_last_update,word_count,trend_direction
13537,content_2c2606c5d176,16,0.930059,347399,4.2,362,104,NaN,down
16959,content_9351f948bf45,17,0.930058,51233,2.0,313,104,NaN,stable
26935,content_37106924f264,18,0.929529,89311,3.4,390,104,NaN,stable
4495,content_f4c93868660b,19,0.929302,87433,3.4,421,104,NaN,stable
17127,content_8818fd6d967f,20,0.929007,83603,3.4,487,104,NaN,down


Fields used in score:
['impressions_90d', 'days_since_last_update', 'avg_position', 'word_count']

Label used in score: False
Trend direction used in score: False

Future-window fields used in score:
[]


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.